Linear Algebra Operations For Machine Learning

ML-Critical Extensions
- Norms & Distances
- Similarity (Cosine, Dot)
- Linear Transformations (Ax)
- Projection

In [1]:
# import

import numpy as np
import torch as th
import tensorflow as tf
import math

In [2]:
# ML-Critical Extensions
# 5. Norms & Distances
# 6. Similarity (Cosine, Dot)
# 7. Linear Transformations (Ax)
# 8. Projection

Norms - A norm = length (magnitude) of a vector
It answers: “How big is this vector?”:
- L2 → sqrt of squares
- L1 → sum of absolute values
- L∞ → max absolute value

In [ ]:
# 5. Norms & Distances
# Norm = length (magnitude) of a vector
# Distance = how far two vectors are apart: d(a,b) = ||a - b||

v = np.array([3.0, 4.0])
a = np.array([1.0, 2.0])
b = np.array([4.0, 6.0])

# --- L2 Norm (Euclidean) --- sqrt of sum of squares
l2_python = (sum(x**2 for x in v)) ** 0.5
l2_numpy  = np.linalg.norm(v)                          # default is L2
l2_torch  = th.norm(th.tensor(v)).item()
l2_tf     = float(tf.norm(tf.constant(v)))

# --- L1 Norm (Manhattan) --- sum of absolute values
l1_python = sum(abs(x) for x in v)
l1_numpy  = np.linalg.norm(v, ord=1)
l1_torch  = th.norm(th.tensor(v), p=1).item()
l1_tf     = float(tf.norm(tf.constant(v), ord=1))

# --- L∞ Norm (Max / Chebyshev) --- max absolute value
linf_python = max(abs(x) for x in v)
linf_numpy  = np.linalg.norm(v, ord=np.inf)
linf_torch  = th.norm(th.tensor(v), p=float('inf')).item()
linf_tf     = float(tf.reduce_max(tf.abs(tf.constant(v))))

# --- Euclidean Distance between a and b ---
dist_python = sum((ai - bi)**2 for ai, bi in zip(a, b)) ** 0.5
dist_numpy  = np.linalg.norm(a - b)
dist_torch  = th.norm(th.tensor(a) - th.tensor(b)).item()
dist_tf     = float(tf.norm(tf.constant(a) - tf.constant(b)))

print(f"Vector v = {v}")
print(f"\n{'Norm':<12} {'Python':>10} {'NumPy':>10} {'PyTorch':>10} {'TF':>10}")
print("-" * 54)
print(f"{'L2 (||v||₂)':<12} {l2_python:>10.4f} {l2_numpy:>10.4f} {l2_torch:>10.4f} {l2_tf:>10.4f}")
print(f"{'L1 (||v||₁)':<12} {l1_python:>10.4f} {l1_numpy:>10.4f} {l1_torch:>10.4f} {l1_tf:>10.4f}")
print(f"{'L∞ (||v||∞)':<12} {linf_python:>10.4f} {linf_numpy:>10.4f} {linf_torch:>10.4f} {linf_tf:>10.4f}")
print()
print(f"Euclidean distance d(a, b) where a={a}, b={b}:")
print(f"  Python={dist_python:.4f}  NumPy={dist_numpy:.4f}  PyTorch={dist_torch:.4f}  TF={dist_tf:.4f}")
print()
print("ML context:")
print("  L2 → most common loss function, weight regularization (L2 reg / Ridge)")
print("  L1 → sparse solutions, feature selection (L1 reg / Lasso)")
print("  L∞ → adversarial robustness bounds")

Squared L2 Norm A Squared L2 Norm = sum of squares of vector elements (no square root)

In [4]:
# Squared L2 Norm = ||v||^2
v = np.array([3, 4])

# Logic: sum of squares (3^2 + 4^2 = 9 + 16 = 25)
squared_norm = np.sum(v**2)

# Using libraries
sq_norm_np = np.linalg.norm(v)**2
sq_norm_th = th.sum(th.tensor([3.0, 4.0])**2)

print(f"Vector: {v}")
print(f"Squared L2 Norm (Logic): {squared_norm}")
print(f"NumPy Result: {sq_norm_np}")
print(f"PyTorch Result: {sq_norm_th.item()}")

Vector: [3 4]
Squared L2 Norm (Logic): 25
NumPy Result: 25.0
PyTorch Result: 25.0


Basis - A basis = a set of vectors that can build (represent) any vector in a space.

In [5]:
# Standard Basis vectors (i and j)
i = np.array([1, 0])
j = np.array([0, 1])

# Any vector can be built from these
# v = 3*i + 4*j
v = 3 * i + 4 * j

print(f"Vector v built from standard basis: {v}")

Vector v built from standard basis: [3 4]


Orthogonal Vectors - Two vectors are orthogonal if they are perpendicular

Orthonormal Vectors -  orthogonal (perpendicular) AND each vector has norm = 1 | Example: v = [1, 0]  | w = [0, 1]

In [6]:
# 1. Orthogonal (Perpendicular) -> Dot product must be 0
v1 = np.array([1, 0])
v2 = np.array([0, 1])
is_orthogonal = np.dot(v1, v2) == 0

# 2. Orthonormal (Orthogonal AND Norm = 1)
norm_v1 = np.linalg.norm(v1)
norm_v2 = np.linalg.norm(v2)
is_orthonormal = is_orthogonal and (math.isclose(norm_v1, 1) and math.isclose(norm_v2, 1))

print(f"Are v1, v2 Orthogonal? {is_orthogonal}")
print(f"Are they Orthonormal? {is_orthonormal}")

# PyTorch Orthonormal Initialization (Real ML usage)
weights = th.empty(3, 3)
th.nn.init.orthogonal_(weights)
print(f"ML Orthonormal Weight Matrix:\n{weights}")

Are v1, v2 Orthogonal? True
Are they Orthonormal? True
ML Orthonormal Weight Matrix:
tensor([[-0.0509, -0.0570, -0.9971],
        [-0.6275, -0.7749,  0.0763],
        [ 0.7770, -0.6295, -0.0037]])


Tiny ML connection
Orthonormal vectors → used in embeddings, PCA
Basis → feature representation
Orthogonality → reduces redundancy

6. Similarity (Cosine, Dot)
Similarity measures how closely two vectors align. Cosine similarity is "scale-invariant," meaning it only cares about the direction, not the magnitude.

In [7]:
# Define two vectors
a = np.array([1, 2, 3])
b = np.array([1, 2, 4])

# Dot Product (Alignment)
dot_sim = np.dot(a, b)

# Cosine Similarity (Angle)
cosine_sim = np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

print(f"Dot Product Similarity: {dot_sim}")
print(f"Cosine Similarity: {cosine_sim:.4f}")

Dot Product Similarity: 17
Cosine Similarity: 0.9915


7. Linear Transformations (Ax)
A matrix $A$ acts as a function that transforms a vector $x$ into a new vector $y$. This is the mathematical foundation of a "Layer" in a Neural Network.

In [8]:
# A transformation matrix (scales X by 2, Y by 0.5)
A = np.array([[2, 0],
              [0, 0.5]])

x = np.array([1, 1])
y = A @ x  # Transform x into y

print(f"Original Vector x: {x}")
print(f"Transformed Vector y: {y}")

Original Vector x: [1 1]
Transformed Vector y: [2.  0.5]


8. Projection
Projection is the process of "dropping" a vector onto a subspace (like a line or plane). It is used in PCA for dimensionality reduction.

In [9]:
v = np.array([2, 3])
L = np.array([4, 0])  # The horizontal axis

# Formula: proj_L(v) = ( (v . L) / ||L||^2 ) * L
projection = (np.dot(v, L) / np.linalg.norm(L) ** 2) * L

print(f"Vector v: {v}")
print(f"Projection of v onto the X-axis: {projection}")

Vector v: [2 3]
Projection of v onto the X-axis: [2. 0.]
